<a href="https://colab.research.google.com/github/mcristinasanchez-ai/TFG_MATEM-TICAS_M.CristinaS-nchezTim-n/blob/main/DE_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import numpy as np
import pandas as pd

# 1. Funciones objetivo que vamos a minimizar
def sphere(x):
    return np.sum(x**2)

def schwefel_1(x):
    return np.sum(np.abs(x)) + np.prod(np.abs(x))

def schwefel_2(x):
    return np.sum([np.sum(x[:i+1])**2 for i in range(len(x))])

def schwefel_3(x):
    return np.max(np.abs(x))

def rosenbrock(x):
    return np.sum(100.0 * (x[1:] - x[:-1]**2.0)**2.0 + (x[:-1] - 1.0)**2.0)

def step(x):
    return np.sum(np.floor(x + 0.5)**2)

def quartic(x):
    return np.sum(np.arange(1, len(x) + 1) * (x**4)) + np.random.random()

def schwefel_4(x):
    return np.sum(-x * np.sin(np.sqrt(np.abs(x))))

def rastrigin(x):
    return np.sum(x**2 - 10 * np.cos(2 * np.pi * x) + 10)

def ackley(x):
    n = float(len(x))
    return -20 * np.exp(-0.2 * np.sqrt(np.sum(x**2) / n)) - np.exp(np.sum(np.cos(2 * np.pi * x)) / n) + 20 + np.exp(1)

funciones_prueba = {
    "Sphere": (sphere, -100, 100),
    "Schwefel 1": (schwefel_1, -10, 10),
    "Schwefel 2": (schwefel_2, -100, 100),
    "Schwefel 3": (schwefel_3, -100, 100),
    "Rosenbrock": (rosenbrock, -30, 30),
    "Step": (step, -100, 100),
    "Quartic": (quartic, -1.28, 1.28),
    "Schwefel 4": (schwefel_4, -500, 500),
    "Rastrigin": (rastrigin, -5.12, 5.12),
    "Ackley": (ackley, -32, 32)
}

# 2. Parámetros
num_vectores = 100 #cantidad de individuos en el espacio de búsqueda
num_iteraciones = 10000 #número de generaciones
dimension = 30  #espacio de búsqueda de 30 dimensiones
F = 0.8  #factor de escala (mutación): controla la amplitud de la diferencia entre vectores
CR = 0.2 #probabilidad de cruce que controla cuantas componentes se modifican, el grado de ingluencia del vector mutante sobre el vector de prueba

num_ejecuciones = 10 #número de ejecuciones por cada función

#Se Almacenan los mejores resultados de cada ejecución
resultados_totales = {}

print("Iniciando optimizaciones...")

for nombre_funcion, (f, limite_inf, limite_sup) in funciones_prueba.items():
    print(f"\nEjecutando 10 veces para: {nombre_funcion}...")
    resultados_funcion = []   #se almacenan los 10 mejores globales de esta función

    for ejecucion in range(num_ejecuciones):
        # 3. Inicialización
        poblacion = np.random.uniform(limite_inf, limite_sup, (num_vectores, dimension)) #matriz donde se colocan los 100 indiviuos de forma aleatoria
        fitness = np.array([f(p) for p in poblacion]) #se evalua su valor

        gmejor_posicion = poblacion[np.argmin(fitness)].copy() #se guarda la mejor posición del individuo
        gmejor_valor = np.min(fitness) #se evalua su valor

        # 4. Bucle principal del algoritmo DE
        for iteracion in range(num_iteraciones):
            for i in range(num_vectores):
                # A. MUTACIÓN
                # Seleccionamos 3 índices aleatorios diferentes entre sí y diferenctes del actual
                idxs = [idx for idx in range(num_vectores) if idx != i]
                r1, r2, r3 = np.random.choice(idxs, 3, replace=False)

                # Para cada individuo se crea el vector mutante mezclando tres indiviuos
                mutante = poblacion[r1] + F * (poblacion[r2] - poblacion[r3])

                #Se usa np.clip para que las partículas no busquen  en zonas no válidas
                #si se superan los límites permitidos
                mutante = np.clip(mutante, limite_inf, limite_sup)

                # B. CRUCE
                # se decide para cada dimensión  si el valor procede del vector
                #mutante o del vector original
                punto_cruce = np.random.rand(dimension) < CR
                # se garantiza que al menos una dimensión cambie
                if not np.any(punto_cruce):
                    punto_cruce[np.random.randint(0, dimension)] = True

                 # se crea el vector de prueba
                vector_prueba = np.where(punto_cruce, mutante, poblacion[i])

                # C. SELECCIÓN
                valor_prueba = f(vector_prueba) #se evalua el vector de prueba

                # Si el nuevo vector es igual o mejor, reemplaza al actual en la población
                if valor_prueba <= fitness[i]:
                    fitness[i] = valor_prueba
                    poblacion[i] = vector_prueba

            # Se actualiza la mejor posición global al final de cada iteración
            current_best_idx = np.argmin(fitness)
            if fitness[current_best_idx] < gmejor_valor:
                gmejor_valor = fitness[current_best_idx]
                gmejor_posicion = poblacion[current_best_idx].copy()

        resultados_funcion.append(gmejor_valor)
        print(f"  Ejecución {ejecucion + 1}: {gmejor_valor:.4e}")

    resultados_totales[nombre_funcion] = resultados_funcion

#CREAMOS LA TABLA CON LOS RESULTADOS DE CADA FUNCIÓN EN LAS 10 Iteraciones y
#Y CALCULAMOS LA MEDIA, MEDIANA Y VARIANZA

# 1. Se crea el Dataframe donde cada columna es una de las funciones de prueba y
#y cada fila son las ejecuiones
df_inicial = pd.DataFrame(resultados_totales)
df_inicial.index = [f"EJECUCIÓN {i+1}" for i in range(num_ejecuciones)]
df_final = df_inicial.copy()

# 2. Se calcula la media, mediana y varianza
df_final.loc['MEDIA'] = df_inicial.mean(axis=0)
df_final.loc['MEDIANA'] = df_inicial.median(axis=0)
df_final.loc['VARIANZA'] = df_inicial.var(axis=0)

# 3. Se muestan los resultados por pantalla y se guardan los resultados en la
#hoja de cáculo
print("\n" + "="*50)
print("TABLA COMPARATIVA FINAL")
print("="*50)
pd.set_option('display.float_format', lambda x: f'{x:.4e}')
print(df_final)
df_final.to_excel("Tabla_comparativaDE.xlsx")
print("\nArchivo 'Tabla_comparativaDE.xlsx' guardado con éxito.")

Iniciando optimizaciones...

Ejecutando 10 veces para: Sphere...
  Ejecución 1: 5.1045e-73
  Ejecución 2: 7.5951e-73
  Ejecución 3: 1.1307e-72
  Ejecución 4: 8.9234e-73
  Ejecución 5: 4.5059e-73
  Ejecución 6: 3.2728e-73
  Ejecución 7: 4.9342e-73
  Ejecución 8: 8.7891e-73
  Ejecución 9: 2.1732e-72
  Ejecución 10: 5.2778e-73

Ejecutando 10 veces para: Schwefel 1...
  Ejecución 1: 2.1923e-44
  Ejecución 2: 1.0122e-44
  Ejecución 3: 1.8092e-44
  Ejecución 4: 3.0780e-44
  Ejecución 5: 2.1949e-44
  Ejecución 6: 1.5432e-44
  Ejecución 7: 1.8056e-44
  Ejecución 8: 1.3573e-44
  Ejecución 9: 1.8965e-44
  Ejecución 10: 9.5619e-45

Ejecutando 10 veces para: Schwefel 2...
  Ejecución 1: 9.7382e+03
  Ejecución 2: 1.0107e+04
  Ejecución 3: 1.0613e+04
  Ejecución 4: 1.0492e+04
  Ejecución 5: 6.8098e+03
  Ejecución 6: 8.7272e+03
  Ejecución 7: 1.0014e+04
  Ejecución 8: 8.3893e+03
  Ejecución 9: 8.8581e+03
  Ejecución 10: 7.3190e+03

Ejecutando 10 veces para: Schwefel 3...
  Ejecución 1: 8.4247e-07
  E